In [16]:
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import os
from tqdm import tqdm

load_dotenv()

# Ensure OPENAI_API_KEY is set in your .env file or as a Colab secret.
# If using Colab secrets, you might need to load it like this:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

df = pd.read_csv("/content/drive/MyDrive/GenAI Final Project Music&Memory/knowledge_base.csv")
chunks = df["text_chunk"].tolist()

def get_embeddings(texts, model="text-embedding-3-small", batch_size=100):
    """Embed a list of texts in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(input=batch, model=model)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
    return np.array(all_embeddings, dtype="float32")

# This will take a few minutes and cost ~$0.10–0.30
embeddings = get_embeddings(chunks)
print(f"Embedding shape: {embeddings.shape}")  # Should be (num_songs, 1536)

# Save embeddings so you don't have to recompute
np.save("/content/drive/MyDrive/GenAI Final Project Music&Memory/embeddings.npy", embeddings)

100%|██████████| 24/24 [00:18<00:00,  1.28it/s]


Embedding shape: (2381, 1536)


In [21]:
import faiss

# Load embeddings if needed
embeddings = np.load("/content/drive/MyDrive/GenAI Final Project Music&Memory/embeddings.npy")

# Build the index
dimension = embeddings.shape[1]  # 1536 for text-embedding-3-small
index = faiss.IndexFlatIP(dimension)  # Inner product (cosine similarity for normalized vectors)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

# Save the index
faiss.write_index(index, "/content/drive/MyDrive/GenAI Final Project Music&Memory/songs.index")

FAISS index built with 2381 vectors


In [19]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.4 MB/s eta 0:00:00


## Summary:

### Data Analysis Key Findings
*   The `faiss-cpu` library was successfully installed.
*   A FAISS index was constructed using 2381 vectors loaded from `embeddings.npy`.
*   The embeddings were normalized using `faiss.normalize_L2` before being added to the `faiss.IndexFlatIP` index.
*   The built index was successfully saved to `/content/drive/MyDrive/GenAI Final Project Music&Memory/songs.index`.

### Insights or Next Steps
*   The created FAISS index can now be used for efficient similarity searches, enabling fast retrieval of similar songs based on their embeddings.
